In [7]:
from pathlib import Path
import numpy as np
import pandas as pd
from datasets import load_dataset
from huggingface_hub import login

In [16]:
import datasets 
dataset = datasets.load_dataset('ucberkeley-dlab/measuring-hate-speech')   
df = dataset['train'].to_pandas()
df.describe()

,comment_id,annotator_id,platform,sentiment,respect,insult,humiliate,status,dehumanize,violence,...,hatespeech,hate_speech_score,infitms,outfitms,annotator_severity,std_err,annotator_infitms,annotator_outfitms,hypothesis,annotator_age
count,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.00000,135556.000000,135556.000000,135556.000000,135556.000000,...,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135451.000000
mean,23530.416138,5567.097812,1.281352,2.954307,2.828875,2.56331,2.278638,2.698575,1.846211,1.052045,...,0.744733,-0.567428,1.034322,1.001052,-0.018817,0.300588,1.007158,1.011841,0.014589,37.910772
std,12387.194125,3230.508937,1.023542,1.231552,1.309548,1.38983,1.370876,0.898500,1.402372,1.345706,...,0.932260,2.380003,0.496867,0.791943,0.487261,0.236380,0.269876,0.675863,0.613006,11.641276
min,1.000000,1.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,...,0.000000,-8.340000,0.100000,0.070000,-1.820000,0.020000,0.390000,0.280000,-1.578693,18.000000
25%,18148.000000,2719.000000,0.000000,2.000000,2.000000,2.00000,1.000000,2.000000,1.000000,0.000000,...,0.000000,-2.330000,0.710000,0.560000,-0.380000,0.030000,0.810000,0.670000,-0.341008,29.000000
50%,20052.000000,5602.500000,1.000000,3.000000,3.000000,3.00000,3.000000,3.000000,2.000000,0.000000,...,0.000000,-0.340000,0.960000,0.830000,-0.020000,0.340000,0.970000,0.850000,0.110405,35.000000
75%,32038.250000,8363.000000,2.000000,4.000000,4.000000,4.00000,3.000000,3.000000,3.000000,2.000000,...,2.000000,1.410000,1.300000,1.220000,0.350000,0.420000,1.170000,1.130000,0.449555,45.000000
max,50070.000000,11142.000000,3.000000,4.000000,4.000000,4.00000,4.000000,4.000000,4.000000,4.000000,...,2.000000,6.300000,5.900000,9.000000,1.360000,1.900000,2.010000,9.000000,0.987511,81.000000


In [17]:
judaism = df.loc[df['target_religion_jewish'] == True, ['comment_id', 'text', 'hate_speech_score']].drop_duplicates(subset='comment_id')
len(judaism)

1874

In [18]:
judaism.to_csv('data/ucberkeley-dlab_target_jewish.csv', index=False)
print(judaism['hate_speech_score'].describe())

count    1874.000000
mean       -0.857556
std         2.006435
min        -7.940000
25%        -2.180000
50%        -0.680000
75%         0.517500
max         5.090000
Name: hate_speech_score, dtype: float64


### Pilot Codebook Labeling

In [5]:
from dotenv import load_dotenv
import os
import anthropic
import json
import csv
import re
import pandas as pd
from config import UNIVERSAL, INPUT, I_1, I_2, I_3, I_4, I_5, I_NO_1, N_1, N_2, N_NO_1, J_1, J_2, J_3, J_NO_1

load_dotenv()
api_key = os.getenv("ANTHROPIC_API_KEY")
client = anthropic.Anthropic(api_key=api_key)

In [8]:
# Sanity Check (Sonnet across 3 runs for intra-model reliability)

judaism = pd.read_csv("data/ucberkeley-dlab_target_jewish.csv")

# test ids and texts
test_ids = [29933, 39476, 40464, 985, 32861, 32448, 27527, 20045]
test_texts = {row['comment_id']: row['text'] for _, row in judaism[judaism['comment_id'].isin(test_ids)].iterrows()}

# prompt blocks (block_name, block_content, is_not)
blocks = [
    ("I_1", I_1, False),
    ("I_2", I_2, False),
    ("I_3", I_3, False),
    ("I_4", I_4, False),
    ("I_5", I_5, False),
    ("I_NO_1", I_NO_1, True),
    ("N_1", N_1, False),
    ("N_2", N_2, False),
    ("N_NO_1", N_NO_1, True),
    ("J_1", J_1, False),
    ("J_2", J_2, False),
    ("J_3", J_3, False),
    ("J_NO_1", J_NO_1, True),
]

MODEL_NAME = "claude-sonnet-4-6"
N_RUNS = 3

# results[run][comment_id][block_name] = [(code_id, label), ...]
results = {r: {} for r in range(1, N_RUNS + 1)}

for run in range(1, N_RUNS + 1):
    for comment_id in test_ids:
        results[run][comment_id] = {}
        text = test_texts[comment_id]

        for block_name, block_content, is_not in blocks:
            is_or_is_not = "is NOT" if is_not else "IS"
            system_text = UNIVERSAL.format(ISorisNOT=is_or_is_not) + block_content
            user_text = INPUT.format(id=comment_id, text=text)

            response = client.messages.create(
                model=MODEL_NAME,
                max_tokens=1024,
                temperature=0,
                system=[
                    {
                        "type": "text",
                        "text": system_text,
                        "cache_control": {"type": "ephemeral"},
                    }
                ],
                messages=[{"role": "user", "content": user_text}],
            )

            raw = response.content[0].text.strip()

            try:
                clean = re.sub(r'```json|```', '', raw).strip()
                data = json.loads(clean)
                parsed = [tuple(pair) for pair in list(data.values())[0]]
                results[run][comment_id][block_name] = parsed
            except Exception as e:
                results[run][comment_id][block_name] = {"parse_error": str(e), "raw": raw}

# save raw json
with open("test/comparative_results.json", "w") as f:
    json.dump(results, f, indent=2)

# flatten to dataframe (comment_id, block, code_id, run, label)
rows = []
for run in range(1, N_RUNS + 1):
    for comment_id, blocks_dict in results[run].items():
        for block_name, labels in blocks_dict.items():
            if isinstance(labels, list):
                for item in labels:
                    if len(item) == 2:
                        code_id, label = item
                        rows.append([comment_id, block_name, code_id, run, label])
                    else:
                        rows.append([comment_id, block_name, "MALFORMED", run, str(item)])
            else:
                rows.append([comment_id, block_name, "PARSE_ERROR", run, str(labels)])

flat_df = pd.DataFrame(rows, columns=["comment_id", "block", "code_id", "run", "label"])
flat_df.to_csv("test/comparative_results_flat.csv", index=False)

print(f"Done. {len(flat_df)} rows saved to comparative_results_flat.csv and comparative_results.json")

Done. 3048 rows saved to comparative_results_flat.csv and comparative_results.json


In [9]:
# Direct comparison across runs

flat_df = pd.read_csv("test/comparative_results_flat.csv")

pivot_df = flat_df.pivot_table(
    index=["comment_id", "block", "code_id"],
    columns="run",
    values="label",
    aggfunc="first"
).reset_index()

pivot_df.to_csv("test/comparative_pivot.csv", index=False)
print(pivot_df.to_string(index=False))

 comment_id  block                 code_id 1 2 3
        985    I_1                  D1HATE N N N
        985    I_1              D1MANIFEST N N N
        985    I_1            D1PERCEPTION A A A
        985    I_1       D2COLLECTIVEBLAME I I I
        985    I_1            D2CONSPIRACY N N N
        985    I_1          D2ISRAELTARGET N N N
        985    I_1            D2STEREOTYPE I I I
        985    I_2               E1RADICAL N N N
        985    I_2              E1VIOLENCE N N N
        985    I_2            E2ALLEGATION I I I
        985    I_2       E2COLLECTIVEPOWER N N N
        985    I_2            E2CONSPIRACY N N N
        985    I_2        E2CONTROLECONOMY N N N
        985    I_2            E2CONTROLGOV N N N
        985    I_2          E2CONTROLMEDIA N N N
        985    I_2          E2CONTROLOTHER N N N
        985    I_2        E2DEHUMANIZATION N N N
        985    I_2              E2DEMONIZE N N N
        985    I_2            E2STEREOTYPE I I I
        985    I_2  

In [10]:
# Intra-model agreement across runs

flat_df = pd.read_csv("test/comparative_results_flat.csv")

def pct_agreement(series_list):
    """Given a list of label-series aligned by index, return % where all match."""
    combined = pd.concat(series_list, axis=1)
    combined.columns = range(len(series_list))
    all_match = combined.apply(lambda row: row.nunique() == 1, axis=1)
    return all_match.mean() * 100

wide = flat_df.pivot_table(
    index=["comment_id", "block", "code_id"],
    columns="run",
    values="label",
    aggfunc="first"
)

run_cols = [c for c in wide.columns]
agreement_pct = pct_agreement([wide[c] for c in run_cols])

print(f"Sonnet: {agreement_pct:.2f}% full agreement across {len(run_cols)} runs")

pd.DataFrame([{"model": "sonnet", "in_model_agreement_pct": round(agreement_pct, 2)}]).to_csv(
    "test/in_model_agreement.csv", index=False
)

Sonnet: 98.72% full agreement across 3 runs


In [11]:
import pandas as pd

df_6 = pd.read_csv("test_0/comparative_results_flat.csv")
df_4 = pd.read_csv("test/comparative_results_flat.csv")

df_6["label_normalized"] = df_6["label"].replace({"O": "I", "C": "I"})

def majority_label(df, label_col="label"):
    return (
        df.groupby(["comment_id", "block", "code_id"])[label_col]
        .agg(lambda x: x.value_counts().idxmax())
        .reset_index()
        .rename(columns={label_col: "label"})
    )

maj_6 = majority_label(df_6, label_col="label_normalized").rename(columns={"label": "label_6code"})
maj_4 = majority_label(df_4).rename(columns={"label": "label_4code"})

merged = maj_6.merge(maj_4, on=["comment_id", "block", "code_id"], how="outer")
merged["agree"] = merged["label_6code"] == merged["label_4code"]

# all runs side by side
pivot_6 = df_6.pivot_table(
    index=["comment_id", "block", "code_id"],
    columns="run",
    values="label_normalized",
    aggfunc="first"
).reset_index()
pivot_6.columns = ["comment_id", "block", "code_id"] + [f"6code_run{c}" for c in pivot_6.columns[3:]]

pivot_4 = df_4.pivot_table(
    index=["comment_id", "block", "code_id"],
    columns="run",
    values="label",
    aggfunc="first"
).reset_index()
pivot_4.columns = ["comment_id", "block", "code_id"] + [f"4code_run{c}" for c in pivot_4.columns[3:]]

all_runs = pivot_6.merge(pivot_4, on=["comment_id", "block", "code_id"], how="outer")
all_runs = all_runs.merge(merged[["comment_id", "block", "code_id", "label_6code", "label_4code", "agree"]], on=["comment_id", "block", "code_id"], how="left")

print("=== All runs side by side with majority labels and agreement ===")
print(all_runs.to_string(index=False))
all_runs.to_csv("test/version_comparison_all_runs.csv", index=False)

overall_pct = merged["agree"].mean() * 100
print(f"\nOverall agreement between 6-code (normalized) and 4-code: {overall_pct:.2f}%")

disagreements = merged[~merged["agree"]]
print(f"\nDisagreements: {len(disagreements)}")
print(disagreements.to_string(index=False))
disagreements.to_csv("test/version_comparison_disagreements.csv", index=False)

def pct_agreement(series_list):
    combined = pd.concat(series_list, axis=1)
    combined.columns = range(len(series_list))
    return combined.apply(lambda row: row.nunique() == 1, axis=1).mean() * 100

for label, df, label_col in [
    ("6-code (normalized)", df_6, "label_normalized"),
    ("4-code", df_4, "label"),
]:
    wide = df.pivot_table(
        index=["comment_id", "block", "code_id"],
        columns="run",
        values=label_col,
        aggfunc="first"
    )
    run_cols = list(wide.columns)
    pct = pct_agreement([wide[c] for c in run_cols])
    print(f"\nIntra-model agreement ({label}): {pct:.2f}% across {len(run_cols)} runs")

=== All runs side by side with majority labels and agreement ===
 comment_id  block                 code_id 6code_run1 6code_run2 6code_run3 4code_run1 4code_run2 4code_run3 label_6code label_4code  agree
        985    I_1                  D1HATE          N          N          N          N          N          N           N           N   True
        985    I_1              D1MANIFEST          N          N          N          N          N          N           N           N   True
        985    I_1            D1PERCEPTION          A          I          A          A          A          A           A           A   True
        985    I_1       D2COLLECTIVEBLAME          I          I          I          I          I          I           I           I   True
        985    I_1            D2CONSPIRACY          N          N          N          N          N          N           N           N   True
        985    I_1          D2ISRAELTARGET          N          N          N          N         

In [ ]:
# categorical distance: adjacent vs cross-bucket disagreements

adjacent_6 = {
    frozenset(["N", "A"]),
    frozenset(["A", "O"]),
    frozenset(["A", "C"]),
    frozenset(["A", "I"]),
    frozenset(["O", "C"]),
    frozenset(["O", "I"]),
    frozenset(["C", "I"]),
    frozenset(["E", "O"]),
    frozenset(["E", "C"]),
    frozenset(["E", "I"]),
    }

adjacent_4 = {
    frozenset(["N", "A"]),
    frozenset(["A", "I"]),
    frozenset(["I", "E"]),
}

def classify_disagreements(df, label_col, adjacent_pairs):
    wide = df.pivot_table(
        index=["comment_id", "block", "code_id"],
        columns="run",
        values=label_col,
        aggfunc="first"
    ).reset_index()
    run_cols = [c for c in wide.columns if c not in ["comment_id", "block", "code_id"]]

    adjacent_count = 0
    cross_count = 0
    cross_examples = []

    for _, row in wide.iterrows():
        labels = list({row[c] for c in run_cols if pd.notna(row[c])})
        if len(labels) > 1:
            # check all pairs of distinct labels seen across runs
            from itertools import combinations
            for l1, l2 in combinations(labels, 2):
                pair = frozenset([l1, l2])
                if pair in adjacent_pairs:
                    adjacent_count += 1
                else:
                    cross_count += 1
                    cross_examples.append({
                        "comment_id": row["comment_id"],
                        "block": row["block"],
                        "code_id": row["code_id"],
                        "labels_seen": str(sorted(labels)),
                    })

    return adjacent_count, cross_count, cross_examples

adj_6, cross_6, cross_ex_6 = classify_disagreements(df_6, "label", adjacent_6)
adj_4, cross_4, cross_ex_4 = classify_disagreements(df_4, "label", adjacent_4)

print("=== Categorical disagreement structure ===")
print(f"\n6-code version:")
print(f"  adjacent-bucket disagreements: {adj_6}")
print(f"  cross-bucket disagreements:    {cross_6}")
if cross_ex_6:
    print("  cross-bucket examples:")
    print(pd.DataFrame(cross_ex_6).to_string(index=False))

print(f"\n4-code version:")
print(f"  adjacent-bucket disagreements: {adj_4}")
print(f"  cross-bucket disagreements:    {cross_4}")
if cross_ex_4:
    print("  cross-bucket examples:")
    print(pd.DataFrame(cross_ex_4).to_string(index=False))

=== Categorical disagreement structure ===

6-code version:
  adjacent-bucket disagreements: 12
  cross-bucket disagreements:    6
  cross-bucket examples:
 comment_id  block               code_id     labels_seen
        985    J_1           A1ESSENTIAL ['A', 'C', 'E']
      20045    I_2             E1RADICAL      ['E', 'N']
      39476 J_NO_1 C11PALESTINIANJUSTICE      ['C', 'N']
      39476 N_NO_1  N16ADVERSEEXPERIENCE ['C', 'E', 'N']
      39476 N_NO_1  N16ADVERSEEXPERIENCE ['C', 'E', 'N']
      40464    I_1          D1PERCEPTION      ['C', 'N']

4-code version:
  adjacent-bucket disagreements: 12
  cross-bucket disagreements:    1
  cross-bucket examples:
 comment_id block           code_id labels_seen
      20045   I_3 E4HOLOCAUSTINTENT  ['I', 'N']


## Full Label